In [1]:
import numpy as np
import pennylane as qml
import itertools
from itertools import combinations
from scipy.optimize import minimize

In [2]:
def aon_hamiltonian(edge):
    edge_length = len(edge)

    minus_terms = []
    plus_terms = []

    for i in edge:
        minus_terms.append(
            qml.Identity(i) - qml.PauliZ(i)
        )

        plus_terms.append(
            qml.Identity(i) + qml.PauliZ(i)
        )

    minus_product = minus_terms[0]
    plus_product = plus_terms[0]

    for op in minus_terms[1:]:
        minus_product = minus_product @ op

    for op in plus_terms[1:]:
        plus_product = plus_product @ op

    return (
        qml.Identity(edge[0])
        - (1 / (2 ** edge_length))
        * (minus_product + plus_product)
    )


def aon_cost_hamiltonian(edges):
    H = 0

    for edge in edges:
        H = H + aon_hamiltonian(edge)

    return H

In [3]:
def dicke_state(n, weight):
    state = np.zeros(2**n, dtype=complex)

    states = list(combinations(range(n), weight))
    amplitude = 1 / np.sqrt(len(states))

    for ones in states:
        bits = np.zeros(n, dtype=int)
        bits[list(ones)] = 1

        index = int("".join(map(str, bits)), 2)

        state[index] = amplitude

    return state

In [4]:
def clique_expansion(hyperedges):
    mixer_edges = set()

    for edge in hyperedges:
        for u, v in itertools.combinations(edge, 2):
            mixer_edges.add(tuple(sorted((u, v))))

    return sorted(mixer_edges)

In [5]:
def swap_mixer(beta, mixer_edges):
    for u, v in mixer_edges:

        qml.S(wires=u)

        qml.SingleExcitationMinus(
            -2 * beta,
            wires=[u, v]
        )

        qml.adjoint(qml.S)(wires=u)

In [6]:
def aon_edge_cost_unitary(gamma, edge):
    edge = list(edge)

    # A size-1 hyperedge can never be cut.
    if len(edge) <= 1:
        return

    controls = edge[:-1]
    target = edge[-1]

    # --------------------------------
    # Phase |11...1>
    # --------------------------------
    qml.ctrl(
        qml.PhaseShift,
        control=controls
    )(
        gamma,
        wires=target
    )

    # --------------------------------
    # Phase |00...0>
    #
    # X maps |00...0> -> |11...1>
    # --------------------------------
    for wire in edge:
        qml.PauliX(wires=wire)

    qml.ctrl(
        qml.PhaseShift,
        control=controls
    )(
        gamma,
        wires=target
    )

    # Undo the X gates
    for wire in edge:
        qml.PauliX(wires=wire)

In [7]:
def aon_cost_layer(gamma, edges):
    for edge in edges:
        aon_edge_cost_unitary(
            gamma,
            edge
        )

In [8]:
def build_qaoa(n, edges):

    dev = qml.device(
        "default.qubit",
        wires=n
    )

    H_C = aon_cost_hamiltonian(edges)

    weight = (n + 1) // 2
    psi0 = dicke_state(n, weight)

    mixer_edges = clique_expansion(edges)

    @qml.qnode(dev)
    def circuit(params):

        gamma = params[0]
        beta = params[1]

        # Balanced initial state
        qml.StatePrep(
            psi0,
            wires=range(n)
        )

        # AoN phase separator
        aon_cost_layer(
            gamma,
            edges
        )

        # Balance-preserving mixer
        swap_mixer(
            beta,
            mixer_edges
        )

        # Expected number of cut hyperedges
        return qml.expval(H_C)

    return circuit

In [9]:
edges = [
    (0, 1, 2),
    (3, 4, 5),
    (0, 3)
]

n = 6

qaoa = build_qaoa(
    n,
    edges
)

In [10]:
params0 = np.array([
    0.0,
    0.0
])

print(qaoa(params0))

2.3999999999999995


In [11]:
result = minimize(
    qaoa,
    x0=np.array([0.5, 0.5]),
    method="COBYLA",
    options={
        "maxiter": 200
    }
)

In [12]:
print("Optimal parameters:")
print(result.x)

print("QAOA expected cost:")
print(result.fun)

Optimal parameters:
[1.35173125 0.26204605]
QAOA expected cost:
1.9987997511647229


In [13]:
def build_probability_circuit(n, edges):

    dev = qml.device(
        "default.qubit",
        wires=n
    )

    weight = (n + 1) // 2
    psi0 = dicke_state(n, weight)

    mixer_edges = clique_expansion(edges)

    @qml.qnode(dev)
    def circuit(params):

        gamma = params[0]
        beta = params[1]

        qml.StatePrep(
            psi0,
            wires=range(n)
        )

        aon_cost_layer(
            gamma,
            edges
        )

        swap_mixer(
            beta,
            mixer_edges
        )

        return qml.probs(wires=range(n))

    return circuit

In [14]:
n = 6

prob_circuit = build_probability_circuit(
    n,
    edges
)

probs = prob_circuit(result.x)

In [16]:
def aon_hyperedge_cost(x, edge):
    vals = [x[i] for i in edge]
    return int(max(vals) != min(vals))

def hypergraph_cut_cost(x, edges):
    return sum(
        aon_hyperedge_cost(x, edge)
        for edge in edges
    )

In [17]:
for i, p in enumerate(probs):

    if p > 1e-6:

        bitstring = format(i, f"0{n}b")

        x = np.array(
            [int(bit) for bit in bitstring]
        )

        cost = hypergraph_cut_cost(
            x,
            edges
        )

        print(
            bitstring,
            "prob =", round(float(p), 4),
            "cost =", cost
        )

000111 prob = 0.0474 cost = 1
001011 prob = 0.0982 cost = 2
001101 prob = 0.0091 cost = 3
001110 prob = 0.0116 cost = 3
010011 prob = 0.107 cost = 2
010101 prob = 0.0057 cost = 3
010110 prob = 0.008 cost = 3
011001 prob = 0.093 cost = 2
011010 prob = 0.1075 cost = 2
011100 prob = 0.0123 cost = 3
100011 prob = 0.0123 cost = 3
100101 prob = 0.1075 cost = 2
100110 prob = 0.093 cost = 2
101001 prob = 0.008 cost = 3
101010 prob = 0.0057 cost = 3
101100 prob = 0.107 cost = 2
110001 prob = 0.0116 cost = 3
110010 prob = 0.0091 cost = 3
110100 prob = 0.0982 cost = 2
111000 prob = 0.0474 cost = 1


In [18]:
def build_qaoa_p2(n, edges):

    dev = qml.device(
        "default.qubit",
        wires=n
    )

    H_C = aon_cost_hamiltonian(edges)

    weight = (n + 1) // 2
    psi0 = dicke_state(n, weight)

    mixer_edges = clique_expansion(edges)

    @qml.qnode(dev)
    def circuit(params):

        gamma1, beta1, gamma2, beta2 = params

        # Initial balanced Dicke state
        qml.StatePrep(
            psi0,
            wires=range(n)
        )

        # Layer 1
        aon_cost_layer(
            gamma1,
            edges
        )

        swap_mixer(
            beta1,
            mixer_edges
        )

        # Layer 2
        aon_cost_layer(
            gamma2,
            edges
        )

        swap_mixer(
            beta2,
            mixer_edges
        )

        return qml.expval(H_C)

    return circuit

In [19]:
edges = [
    (0, 1, 2),
    (3, 4, 5),
    (0, 3)
]

n = 6

qaoa_p2 = build_qaoa_p2(
    n,
    edges
)

In [20]:
x0 = np.array([
    0.5, 0.5,
    0.5, 0.5
])

result_p2 = minimize(
    qaoa_p2,
    x0=x0,
    method="COBYLA",
    options={
        "maxiter": 500
    }
)

print("Optimal parameters:")
print(result_p2.x)

print("QAOA p=2 expected cost:")
print(result_p2.fun)

Optimal parameters:
[0.54689425 0.35251323 1.65053646 0.24656474]
QAOA p=2 expected cost:
1.8915784591786866


In [21]:
def build_probability_circuit_p2(n, edges):

    dev = qml.device(
        "default.qubit",
        wires=n
    )

    weight = (n + 1) // 2
    psi0 = dicke_state(n, weight)

    mixer_edges = clique_expansion(edges)

    @qml.qnode(dev)
    def circuit(params):

        gamma1, beta1, gamma2, beta2 = params

        qml.StatePrep(
            psi0,
            wires=range(n)
        )

        aon_cost_layer(
            gamma1,
            edges
        )

        swap_mixer(
            beta1,
            mixer_edges
        )

        aon_cost_layer(
            gamma2,
            edges
        )

        swap_mixer(
            beta2,
            mixer_edges
        )

        return qml.probs(
            wires=range(n)
        )

    return circuit

In [22]:
prob_circuit_p2 = build_probability_circuit_p2(
    n,
    edges
)

probs_p2 = prob_circuit_p2(
    result_p2.x
)

In [23]:
results = []

for i, p in enumerate(probs_p2):

    if p > 1e-6:

        bitstring = format(i, f"0{n}b")

        x = np.array(
            [int(bit) for bit in bitstring]
        )

        cost = hypergraph_cut_cost(
            x,
            edges
        )

        results.append(
            (bitstring, float(p), cost)
        )

results.sort(
    key=lambda x: x[1],
    reverse=True
)

for bitstring, p, cost in results:

    print(
        bitstring,
        "prob =", round(p, 4),
        "cost =", cost
    )

011010 prob = 0.1085 cost = 2
100101 prob = 0.1085 cost = 2
010011 prob = 0.1085 cost = 2
101100 prob = 0.1085 cost = 2
011001 prob = 0.1042 cost = 2
100110 prob = 0.1042 cost = 2
001011 prob = 0.1041 cost = 2
110100 prob = 0.1041 cost = 2
000111 prob = 0.0644 cost = 1
111000 prob = 0.0644 cost = 1
011100 prob = 0.008 cost = 3
100011 prob = 0.008 cost = 3
001110 prob = 0.0008 cost = 3
110001 prob = 0.0008 cost = 3
001101 prob = 0.0006 cost = 3
110010 prob = 0.0006 cost = 3
010110 prob = 0.0005 cost = 3
101001 prob = 0.0005 cost = 3
010101 prob = 0.0004 cost = 3
101010 prob = 0.0004 cost = 3


In [24]:
cost_mass = {}

for i, p in enumerate(probs_p2):

    if p > 1e-10:

        bitstring = format(i, f"0{n}b")
        x = np.array([int(b) for b in bitstring])

        cost = hypergraph_cut_cost(
            x,
            edges
        )

        cost_mass[cost] = (
            cost_mass.get(cost, 0)
            + float(p)
        )

for cost in sorted(cost_mass):

    print(
        "cost =", cost,
        "total probability =",
        round(cost_mass[cost], 4)
    )

cost = 1 total probability = 0.1289
cost = 2 total probability = 0.8507
cost = 3 total probability = 0.0204


In [25]:
def get_edge_list(P):
    edges = []

    for e_idx in range(P.shape[1]):
        edge = tuple(
            int(v)
            for v in np.flatnonzero(P[:, e_idx] != 0)
        )
        edges.append(edge)

    return edges

In [33]:
def run_qaoa_instance(P, maxiter=500):

    edges = get_edge_list(P)
    n = P.shape[0]

    print("n =", n)
    print("m =", len(edges))
    print("edge sizes =", [len(e) for e in edges])

    # -------------------------
    # Build p=2 QAOA circuit
    # -------------------------
    qaoa = build_qaoa_p2(
        n,
        edges
    )

    x0 = np.array([
        0.5, 0.5,
        0.5, 0.5
    ])

    result = minimize(
        qaoa,
        x0=x0,
        method="COBYLA",
        options={
            "maxiter": maxiter
        }
    )

    print("\nOptimal parameters:")
    print(result.x)

    print("\nOptimized expected cost:")
    print(result.fun)

    # -------------------------
    # Get optimized distribution
    # -------------------------
    prob_circuit = build_probability_circuit_p2(
        n,
        edges
    )

    probs = prob_circuit(
        result.x
    )

    # -------------------------
    # Find best observed state
    # -------------------------
    best_cost = np.inf
    best_states = []
    best_prob = 0.0

    for i, p in enumerate(probs):

        if p > 1e-10:

            bitstring = format(
                i,
                f"0{n}b"
            )

            x = np.array(
                [int(bit) for bit in bitstring]
            )

            cost = hypergraph_cut_cost(
                x,
                edges
            )

            if cost < best_cost:

                best_cost = cost
                best_states = [bitstring]
                best_prob = float(p)

            elif cost == best_cost:

                best_states.append(bitstring)
                best_prob += float(p)

    print("\nBest cost in QAOA distribution:")
    print(best_cost)

    print("Best states:")
    print(best_states)

    print("Total probability of best-cost states:")
    print(best_prob)

    return result

In [34]:
graph_type = "poisson_16_12_3"

for g in [1, 2, 3]:

    instance = f"{g:02d}"

    print("\n-------------------------")
    print("Instance:", instance)
    print("-------------------------")

    P = np.load(
        f"data/{graph_type}/P/{instance}.npy"
    )

    result = run_qaoa_instance(
        P,
        maxiter=500
    )


-------------------------
Instance: 01
-------------------------
n = 12
m = 15
edge sizes = [4, 3, 3, 2, 2, 2, 6, 4, 4, 3, 3, 5, 3, 3, 5]

Optimal parameters:
[0.20121207 0.7561841  0.67397188 0.4165335 ]

Optimized expected cost:
11.956149140604529

Best cost in QAOA distribution:
6
Best states:
['001100110110', '110011001001']
Total probability of best-cost states:
0.0032962649472876504

-------------------------
Instance: 02
-------------------------
n = 12
m = 16
edge sizes = [2, 2, 3, 3, 3, 2, 4, 2, 2, 4, 4, 3, 3, 5, 2, 2]

Optimal parameters:
[1.52801365 1.50257514 0.08272984 0.43136438]

Optimized expected cost:
11.467666856941246

Best cost in QAOA distribution:
6
Best states:
['011100000111', '100011111000']
Total probability of best-cost states:
0.004424852978799916

-------------------------
Instance: 03
-------------------------
n = 12
m = 16
edge sizes = [2, 3, 2, 3, 4, 3, 3, 3, 2, 4, 3, 3, 3, 3, 4, 4]

Optimal parameters:
[ 0.27007694  0.08543597  0.69296033 -0.38919031]

In [35]:
import numpy as np
from pathlib import Path


def max_hyperedge_size(P):
    # columns are hyperedges
    edge_sizes = np.count_nonzero(P, axis=0)

    return edge_sizes.max()


def check_dataset(folder):
    folder = Path(folder)

    for file in sorted(folder.glob("*.npy")):
        P = np.load(file)

        max_size = max_hyperedge_size(P)

        print(
            f"{file.name:25s} "
            f"vertices={P.shape[0]:2d} "
            f"edges={P.shape[1]:2d} "
            f"max_edge={max_size}"
        )

In [36]:
print("\nPOISSON μ=3")
check_dataset("data/poisson_16_12_3/P")

print("\nPOISSON μ=4")
check_dataset("data/poisson_16_12_4/P")

print("\nPOISSON μ=5")
check_dataset("data/poisson_16_12_5/P")

print("\nCONGRESS")
check_dataset("data/congress-bills/numpy_20")


POISSON μ=3
01.npy                    vertices=12 edges=15 max_edge=6
02.npy                    vertices=12 edges=16 max_edge=5
03.npy                    vertices=12 edges=16 max_edge=4
04.npy                    vertices=12 edges=16 max_edge=5
05.npy                    vertices=12 edges=16 max_edge=4
06.npy                    vertices=12 edges=16 max_edge=5
07.npy                    vertices=12 edges=16 max_edge=5
08.npy                    vertices=12 edges=15 max_edge=4
09.npy                    vertices=12 edges=16 max_edge=7
10.npy                    vertices=12 edges=16 max_edge=7
11.npy                    vertices=12 edges=16 max_edge=4
12.npy                    vertices=12 edges=15 max_edge=5
13.npy                    vertices=12 edges=16 max_edge=5
14.npy                    vertices=12 edges=16 max_edge=5
15.npy                    vertices=12 edges=16 max_edge=5
16.npy                    vertices=12 edges=16 max_edge=5
17.npy                    vertices=12 edges=16 max_edge=4
1